# 🛒 Customer Segmentation with RFM Analysis

> **Business Problem:** The marketing team is spending budget treating all customers the same way.
> By segmenting customers based on their purchasing behavior, we can tailor marketing actions
> to maximize ROI and reduce churn.

**Dataset:** UCI Online Retail II — real transactions from a UK e-commerce store (2009–2011)  
**Download:** https://archive.ics.uci.edu/dataset/502/online+retail+ii

---

### 📋 Notebook Structure
1. Setup & Data Loading
2. Data Cleaning
3. RFM Calculation
4. RFM Scoring & Manual Segmentation
5. K-Means Clustering
6. Visualization
7. Business Recommendations

## 1. Setup & Data Loading

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')
import os

# ── Style ─────────────────────────────────────────────────────────────────────
plt.style.use('seaborn-v0_8-whitegrid')
PALETTE = {
    'Champions':          '#2ECC71',
    'Loyal Customers':    '#3498DB',
    'Potential Loyalists':'#9B59B6',
    'New Customers':      '#F39C12',
    'At Risk':            '#E74C3C',
    'Cannot Lose Them':   '#C0392B',
    'Hibernating':        '#95A5A6',
    'Lost':               '#7F8C8D'
}

os.makedirs('plots', exist_ok=True)
print('✅ Libraries loaded successfully!')

In [ ]:
# ── Load Data ─────────────────────────────────────────────────────────────────
# The dataset has two sheets: 'Year 2009-2010' and 'Year 2010-2011'
# We use the larger sheet for this analysis

FILE_PATH = 'online_retail_II.xlsx'   # <-- update path if needed

df = pd.read_excel(FILE_PATH, sheet_name='Year 2010-2011')

print(f'Shape      : {df.shape}')
print(f'Columns    : {df.columns.tolist()}')
print(f'Date range : {df["InvoiceDate"].min()} → {df["InvoiceDate"].max()}')
df.head()

## 2. Data Cleaning

In [ ]:
# ── Inspect ───────────────────────────────────────────────────────────────────
print('=== Missing Values ===')
print(df.isnull().sum())
print(f'\nTotal rows before cleaning: {len(df):,}')

In [ ]:
# ── Clean ─────────────────────────────────────────────────────────────────────

# 1. Drop rows without CustomerID (can't build RFM without it)
df.dropna(subset=['Customer ID'], inplace=True)

# 2. Remove cancellations — InvoiceNo starting with 'C'
df = df[~df['Invoice'].astype(str).str.startswith('C')]

# 3. Remove impossible values
df = df[(df['Quantity'] > 0) & (df['Price'] > 0)]

# 4. Feature engineering
df['TotalPrice']  = df['Quantity'] * df['Price']
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['Customer ID'] = df['Customer ID'].astype(int)

print(f'Total rows after cleaning: {len(df):,}')
print(f'Unique customers         : {df["Customer ID"].nunique():,}')
print(f'Unique invoices          : {df["Invoice"].nunique():,}')
df.head()

## 3. RFM Calculation

| Metric | Definition | Business meaning |
|--------|-----------|------------------|
| **Recency** | Days since last purchase | Are they still engaged? |
| **Frequency** | Number of unique orders | Are they habitual buyers? |
| **Monetary** | Total amount spent | How valuable are they? |

In [ ]:
# ── RFM Calculation ───────────────────────────────────────────────────────────
# Reference date = 1 day after the last transaction in the dataset
REFERENCE_DATE = df['InvoiceDate'].max() + pd.Timedelta(days=1)
print(f'Reference date: {REFERENCE_DATE.date()}')

rfm = df.groupby('Customer ID').agg(
    Recency   = ('InvoiceDate', lambda x: (REFERENCE_DATE - x.max()).days),
    Frequency = ('Invoice',     'nunique'),
    Monetary  = ('TotalPrice',  'sum')
).reset_index()

rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

print(f'\nRFM table shape: {rfm.shape}')
print('\nStatistical Summary:')
rfm[['Recency', 'Frequency', 'Monetary']].describe().round(2)

In [ ]:
# ── Visualise RFM Distributions ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('RFM Metric Distributions', fontsize=15, fontweight='bold', y=1.02)

metrics = [
    ('Recency',   '#E74C3C', 'Days Since Last Purchase'),
    ('Frequency', '#3498DB', 'Number of Orders'),
    ('Monetary',  '#2ECC71', 'Total Spend (£)'),
]

for ax, (metric, color, xlabel) in zip(axes, metrics):
    ax.hist(rfm[metric], bins=40, color=color, alpha=0.8, edgecolor='white', linewidth=0.5)
    ax.axvline(rfm[metric].median(), color='black', linestyle='--',
               linewidth=1.5, label=f'Median: {rfm[metric].median():.0f}')
    ax.set_title(metric, fontsize=12, fontweight='bold')
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Number of Customers')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('plots/01_rfm_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved → plots/01_rfm_distributions.png')

## 4. RFM Scoring & Manual Segmentation

Each metric is split into **5 equal quantile groups (1–5)**.  
- For **Recency**: lower days = better, so score is **reversed** (5 = most recent)  
- For **Frequency** and **Monetary**: higher = better (5 = highest)

In [ ]:
# ── RFM Scoring ───────────────────────────────────────────────────────────────
rfm['R_Score'] = pd.qcut(rfm['Recency'],
                          q=5, labels=[5, 4, 3, 2, 1])  # reversed: lower days = higher score
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'),
                          q=5, labels=[1, 2, 3, 4, 5])
rfm['M_Score'] = pd.qcut(rfm['Monetary'],
                          q=5, labels=[1, 2, 3, 4, 5])

# Combined string score (e.g. '555' = best customer)
rfm['RFM_Score'] = (rfm['R_Score'].astype(str)
                    + rfm['F_Score'].astype(str)
                    + rfm['M_Score'].astype(str))

# Numeric total for quick sorting
rfm['RFM_Total'] = (rfm['R_Score'].astype(int)
                    + rfm['F_Score'].astype(int)
                    + rfm['M_Score'].astype(int))

print('✅ Scoring complete — sample output:')
rfm[['CustomerID', 'Recency', 'Frequency', 'Monetary',
     'R_Score', 'F_Score', 'M_Score', 'RFM_Score']].head(10)

In [ ]:
# ── Segment Mapping ───────────────────────────────────────────────────────────
def assign_segment(row):
    r = int(row['R_Score'])
    f = int(row['F_Score'])
    m = int(row['M_Score'])

    if r >= 4 and f >= 4 and m >= 4:
        return 'Champions'
    elif r >= 3 and f >= 3:
        return 'Loyal Customers'
    elif r >= 4 and f <= 2:
        return 'New Customers'
    elif r >= 3 and f <= 2:
        return 'Potential Loyalists'
    elif r <= 2 and f >= 4:
        return 'Cannot Lose Them'
    elif r <= 2 and f >= 2 and m >= 3:
        return 'At Risk'
    elif r <= 2 and f <= 2 and m <= 2:
        return 'Lost'
    else:
        return 'Hibernating'

rfm['Segment'] = rfm.apply(assign_segment, axis=1)

print('=== Segment Distribution ===')
print(rfm['Segment'].value_counts())

In [ ]:
# ── Segment Summary Table ─────────────────────────────────────────────────────
segment_summary = (
    rfm.groupby('Segment')
    .agg(
        Customer_Count = ('CustomerID',  'count'),
        Avg_Recency    = ('Recency',     'mean'),
        Avg_Frequency  = ('Frequency',   'mean'),
        Avg_Monetary   = ('Monetary',    'mean'),
        Total_Revenue  = ('Monetary',    'sum')
    )
    .round(2)
    .reset_index()
)

total_revenue = segment_summary['Total_Revenue'].sum()
segment_summary['Revenue_Share_%'] = (
    segment_summary['Total_Revenue'] / total_revenue * 100
).round(1)

segment_summary.sort_values('Total_Revenue', ascending=False)

In [ ]:
# ── Visualise Segments ────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Customer Segmentation — RFM Analysis', fontsize=16,
             fontweight='bold', y=1.01)

# --- Plot 1: Customer count per segment (horizontal bar) ---
ss = segment_summary.sort_values('Customer_Count')
bar_colors = [PALETTE.get(s, '#95A5A6') for s in ss['Segment']]
axes[0, 0].barh(ss['Segment'], ss['Customer_Count'], color=bar_colors, edgecolor='white')
axes[0, 0].set_title('Number of Customers per Segment', fontweight='bold')
axes[0, 0].set_xlabel('Customer Count')
for i, v in enumerate(ss['Customer_Count']):
    axes[0, 0].text(v + 5, i, f'{v:,}', va='center', fontsize=9)

# --- Plot 2: Revenue share pie chart ---
ss2 = segment_summary.sort_values('Total_Revenue', ascending=False)
pie_colors = [PALETTE.get(s, '#95A5A6') for s in ss2['Segment']]
axes[0, 1].pie(
    ss2['Total_Revenue'],
    labels=ss2['Segment'],
    colors=pie_colors,
    autopct='%1.1f%%',
    startangle=140,
    textprops={'fontsize': 9}
)
axes[0, 1].set_title('Revenue Share by Segment', fontweight='bold')

# --- Plot 3: Average spend per customer ---
ss3 = segment_summary.sort_values('Avg_Monetary')
bar_colors3 = [PALETTE.get(s, '#95A5A6') for s in ss3['Segment']]
axes[1, 0].barh(ss3['Segment'], ss3['Avg_Monetary'], color=bar_colors3, edgecolor='white')
axes[1, 0].set_title('Average Spend per Customer (£)', fontweight='bold')
axes[1, 0].set_xlabel('Average Monetary Value (£)')
for i, v in enumerate(ss3['Avg_Monetary']):
    axes[1, 0].text(v + 5, i, f'£{v:,.0f}', va='center', fontsize=9)

# --- Plot 4: Recency vs Frequency scatter (coloured by segment) ---
scatter_colors = [PALETTE.get(s, '#95A5A6') for s in rfm['Segment']]
axes[1, 1].scatter(rfm['Recency'], rfm['Frequency'],
                   c=scatter_colors, alpha=0.45, s=18)
axes[1, 1].set_title('Recency vs Frequency (by Segment)', fontweight='bold')
axes[1, 1].set_xlabel('Recency (days since last purchase)')
axes[1, 1].set_ylabel('Frequency (number of orders)')
legend_handles = [mpatches.Patch(color=v, label=k) for k, v in PALETTE.items()]
axes[1, 1].legend(handles=legend_handles, fontsize=8,
                  loc='upper right', ncol=2, framealpha=0.8)

plt.tight_layout()
plt.savefig('plots/02_segment_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved → plots/02_segment_analysis.png')

## 5. K-Means Clustering

While manual segmentation is business-friendly, **K-Means** finds natural data clusters without predefined rules.
Comparing both approaches validates our segment definitions.

In [ ]:
# ── Normalise RFM ─────────────────────────────────────────────────────────────
scaler    = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[['Recency', 'Frequency', 'Monetary']])
print('Scaling complete. Sample (first 3 rows):')
print(rfm_scaled[:3].round(3))

In [ ]:
# ── Elbow Method + Silhouette Score ───────────────────────────────────────────
K_range      = range(2, 11)
inertias     = []
silhouettes  = []

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(rfm_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(rfm_scaled, labels))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

# Elbow
ax1.plot(K_range, inertias, 'bo-', markersize=8, linewidth=2)
ax1.set_title('Elbow Method — Inertia vs K', fontweight='bold')
ax1.set_xlabel('Number of Clusters (K)')
ax1.set_ylabel('Inertia (within-cluster sum of squares)')

# Silhouette
ax2.plot(K_range, silhouettes, 'rs-', markersize=8, linewidth=2)
ax2.set_title('Silhouette Score vs K', fontweight='bold')
ax2.set_xlabel('Number of Clusters (K)')
ax2.set_ylabel('Silhouette Score (higher = better)')

best_k = list(K_range)[silhouettes.index(max(silhouettes))]
ax2.axvline(best_k, color='green', linestyle='--', label=f'Best K = {best_k}')
ax2.legend()

plt.tight_layout()
plt.savefig('plots/03_optimal_k.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'💡 Suggested K based on silhouette score: {best_k}')

In [ ]:
# ── Fit Final K-Means Model ───────────────────────────────────────────────────
OPTIMAL_K = 4   # Adjust based on elbow + silhouette charts above

kmeans = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10)
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)

# Profile each cluster
cluster_profile = (
    rfm.groupby('Cluster')[['Recency', 'Frequency', 'Monetary']]
    .mean()
    .round(1)
)
cluster_profile['Customer_Count'] = rfm.groupby('Cluster')['CustomerID'].count()

print('=== Cluster Profiles ===')
print(cluster_profile)
print('\nHint: Low Recency + High Freq/Monetary = Champions cluster')

In [ ]:
# ── PCA Visualisation of Clusters ─────────────────────────────────────────────
pca     = PCA(n_components=2)
rfm_pca = pca.fit_transform(rfm_scaled)

cmap_colors = ['#3498DB', '#E74C3C', '#2ECC71', '#F39C12']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# K-Means clusters
for cluster_id in sorted(rfm['Cluster'].unique()):
    mask = rfm['Cluster'] == cluster_id
    axes[0].scatter(rfm_pca[mask, 0], rfm_pca[mask, 1],
                    c=cmap_colors[cluster_id], alpha=0.5, s=18,
                    label=f'Cluster {cluster_id}')

axes[0].set_title('K-Means Clusters (PCA projection)', fontweight='bold')
axes[0].set_xlabel(f'PC1 — {pca.explained_variance_ratio_[0]*100:.1f}% variance')
axes[0].set_ylabel(f'PC2 — {pca.explained_variance_ratio_[1]*100:.1f}% variance')
axes[0].legend(markerscale=2, fontsize=9)

# Manual segments
for segment, color in PALETTE.items():
    mask = rfm['Segment'] == segment
    if mask.sum() > 0:
        axes[1].scatter(rfm_pca[mask, 0], rfm_pca[mask, 1],
                        c=color, alpha=0.5, s=18, label=segment)

axes[1].set_title('Manual RFM Segments (PCA projection)', fontweight='bold')
axes[1].set_xlabel(f'PC1 — {pca.explained_variance_ratio_[0]*100:.1f}% variance')
axes[1].set_ylabel(f'PC2 — {pca.explained_variance_ratio_[1]*100:.1f}% variance')
axes[1].legend(markerscale=2, fontsize=8, ncol=2)

plt.tight_layout()
plt.savefig('plots/04_pca_clusters.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved → plots/04_pca_clusters.png')

## 6. RFM Heatmap

In [ ]:
# ── R vs F Heatmap coloured by average Monetary value ─────────────────────────
rfm_heatmap = (
    rfm.groupby(['R_Score', 'F_Score'])['Monetary']
    .mean()
    .reset_index()
    .pivot(index='R_Score', columns='F_Score', values='Monetary')
    .iloc[::-1]  # flip so R=5 is at the top
)

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(
    rfm_heatmap,
    annot=True, fmt='.0f', cmap='YlOrRd',
    linewidths=0.5, linecolor='white',
    cbar_kws={'label': 'Avg Revenue (£)', 'shrink': 0.8},
    ax=ax
)
ax.set_title('Average Revenue by R-Score × F-Score\n(Higher R = more recent; Higher F = more frequent)',
             fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('Frequency Score (F)', fontsize=11)
ax.set_ylabel('Recency Score (R)', fontsize=11)

plt.tight_layout()
plt.savefig('plots/05_rfm_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved → plots/05_rfm_heatmap.png')

## 7. Business Recommendations

> **Insight first, then action.** Each segment needs a different response.

In [ ]:
# ── Business Action Plan ──────────────────────────────────────────────────────
recommendations = [
    {
        'Segment':      'Champions',
        'Strategy':     'Reward & Retain',
        'Tactics':      'Exclusive loyalty perks, early access to new products, referral program invites',
        'Goal':         'Turn them into brand ambassadors'
    },
    {
        'Segment':      'Loyal Customers',
        'Strategy':     'Upsell & Deepen',
        'Tactics':      'Premium product recommendations, loyalty points multipliers, personalised thank-you notes',
        'Goal':         'Increase average order value (AOV)'
    },
    {
        'Segment':      'Potential Loyalists',
        'Strategy':     'Nurture & Convert',
        'Tactics':      'Onboarding email series, first repeat-purchase discount, educational content',
        'Goal':         'Graduate to Loyal Customers within 90 days'
    },
    {
        'Segment':      'New Customers',
        'Strategy':     'Activate Fast',
        'Tactics':      'Welcome flow, quick-win product bundle, social proof',
        'Goal':         'Secure second purchase within 30 days'
    },
    {
        'Segment':      'At Risk',
        'Strategy':     'Win-Back Campaign',
        'Tactics':      '"We miss you" personalised email, limited-time offer, survey on why they left',
        'Goal':         'Reactivate before they go permanently silent'
    },
    {
        'Segment':      'Cannot Lose Them',
        'Strategy':     'High-Priority Re-engagement',
        'Tactics':      'Dedicated account manager outreach, VIP retention offer, direct phone/email contact',
        'Goal':         'These are high-frequency buyers — losing them hurts revenue immediately'
    },
    {
        'Segment':      'Hibernating',
        'Strategy':     'Low-Cost Reactivation',
        'Tactics':      'Seasonal or category-relevant email, small discount if no engagement after 2 nudges',
        'Goal':         'Reactivate the ones who respond; accept the others as churned'
    },
    {
        'Segment':      'Lost',
        'Strategy':     'Minimal Spend',
        'Tactics':      'One-time deep discount or free shipping offer. No response → suppress from lists',
        'Goal':         'Recover the few who are still reachable; free up budget from the rest'
    },
]

rec_df = pd.DataFrame(recommendations)
print('=== Business Action Plan ===')
for _, row in rec_df.iterrows():
    print(f"\n{'─'*60}")
    print(f"  Segment  : {row['Segment']}")
    print(f"  Strategy : {row['Strategy']}")
    print(f"  Tactics  : {row['Tactics']}")
    print(f"  Goal     : {row['Goal']}")

In [ ]:
# ── Final: Export clean RFM table ─────────────────────────────────────────────
export_cols = ['CustomerID', 'Recency', 'Frequency', 'Monetary',
               'R_Score', 'F_Score', 'M_Score', 'RFM_Score', 'Segment', 'Cluster']
rfm[export_cols].to_csv('rfm_output.csv', index=False)
print('✅ RFM results exported → rfm_output.csv')
print(f'\nFinal dataset: {len(rfm):,} customers across {rfm["Segment"].nunique()} segments')
rfm[export_cols].head(10)

---

## 📌 Key Takeaways

| Finding | Business Implication |
|---------|---------------------|
| **Champions + Loyal** drive the majority of revenue from a minority of customers | Focus retention budget here first |
| **At Risk** segment has previously high frequency, now disengaging | High urgency — win-back campaigns |
| **Lost** customers are numerous but low-value | Minimal marketing spend justified |
| RFM Month 1 drop-off indicates weak onboarding | Fix new customer experience for compounding gains |

> *Built with Python • pandas • scikit-learn • matplotlib/seaborn*